# Pronomen-Extraktion für die Koreferenz-Evaluation



**Ordnerkontext:** Das Notebook liegt in `00_werkzeuge/`, liest aus
`02_quelldaten/` und schreibt die Vorlage nach `02_quelldaten/` — die
Pfade in Schritt 0 sind entsprechend relativ gesetzt.

## Konfiguration


In [ ]:
from pathlib import Path

BASIS       = Path("..") / "02_quelldaten"
GOLD_CONLL  = BASIS / "goldstandard_v1.conll"
AUSGABE_CSV = BASIS / "goldstandard_v1_pronomen.csv"
SPACY_MODELL = "de_core_news_lg"

print(f"Quelle: {GOLD_CONLL}")
print(f"Ziel  : {AUSGABE_CSV}")
assert GOLD_CONLL.exists(), (
    f"{GOLD_CONLL} nicht gefunden — läuft das Notebook aus 00_werkzeuge/? "
    "Sonst BASIS in dieser Zelle absolut setzen.")

Quelle: ../02_quelldaten/goldstandard_v1.conll
Ziel  : ../02_quelldaten/goldstandard_v1_pronomen.csv


## Schritt 1 — Filterkriterien definieren


In [ ]:
# Personalpronomen 3. Person (die klassischen Koreferenz-Fälle)
PERSONAL = {"er", "sie", "es", "ihm", "ihn", "ihr", "ihnen"}

# Demonstrativ verwendete Kurzformen ("der freute sich")
DEMONSTRATIV = {"der", "die", "das", "dem", "den",
                "derselbe", "dieselbe", "dasselbe",
                "demselben", "denselben", "derselben"}

print(f"Personalpronomen: {len(PERSONAL)} Formen")
print(f"Demonstrativformen: {len(DEMONSTRATIV)} Formen")

## Schritt 2 — Goldsätze strukturerhaltend einlesen


In [ ]:
def lese_bloecke(pfad):
    saetze, meta, start, offset, cur = [], {}, 0, 0, []

    def flush():
        nonlocal cur, offset
        if cur:
            saetze.append(dict(sammlung=meta.get("sammlung", "?"),
                               erzaehlung=meta.get("erzaehlung", "?"),
                               tranche=meta.get("tranche", "?"),
                               block=meta.get("block", "?"),
                               satzindex=start + offset,
                               tokens=cur))
            offset += 1
            cur = []

    for roh in open(pfad, encoding="utf-8"):
        z = roh.rstrip("\n")
        if z.startswith("#"):
            if "=" in z:
                k, _, v = z.lstrip("#").partition("=")
                k, v = k.strip(), v.strip()
                if k == "erzaehlung":          # neuer Block beginnt
                    flush()
                    meta, offset = {}, 0
                meta[k] = v
                if k == "satzindizes":
                    start = int(v.split("-")[0])
            continue
        if not z.strip():
            flush()
            continue
        cur.append(z.split("\t")[0])
    flush()
    return saetze

saetze = lese_bloecke(GOLD_CONLL)
print(f"Eingelesen: {len(saetze)} Sätze aus "
      f"{len({s['erzaehlung'] for s in saetze})} Erzählungen")

## Schritt 3 — Pronomen mit spaCy identifizieren


In [ ]:
import spacy
from spacy.tokens import Doc

nlp = spacy.load(SPACY_MODELL,
                 disable=["lemmatizer", "attribute_ruler", "ner", "parser"])

from collections import defaultdict
block_saetze = defaultdict(list)
for s in saetze:
    block_saetze[(s["erzaehlung"], s["block"])].append(s)
for schluessel in block_saetze:
    block_saetze[schluessel].sort(key=lambda x: x["satzindex"])

zeilen = []
for s in saetze:
    doc = Doc(nlp.vocab, words=s["tokens"])
    for _, proc in nlp.pipeline:                # tagger + morphologizer
        doc = proc(doc)
    for t in doc:
        if t.pos_ != "PRON":
            continue
        m = t.morph
        # Ausschluss: Relativ- und Reflexivpronomen
        if m.get("PronType") == ["Rel"] or m.get("Reflex") == ["Yes"]:
            continue
        wort = t.text.lower()
        ist_personal = (m.get("PronType") == ["Prs"]
                        and m.get("Person") == ["3"]
                        and wort in PERSONAL)
        ist_demonstrativ = (m.get("PronType") == ["Dem"]
                            and wort in DEMONSTRATIV)
        if not (ist_personal or ist_demonstrativ):
            continue
        markiert = (s["tokens"][:t.i] + [f"**{t.text}**"]
                    + s["tokens"][t.i + 1:])
        teile = []
        for nachbar in block_saetze[(s["erzaehlung"], s["block"])]:
            if nachbar["satzindex"] == s["satzindex"]:
                teile.append(" ".join(markiert))
            else:
                teile.append(" ".join(nachbar["tokens"]))
        block_text = "  ".join(teile)
        zeilen.append(dict(
            sammlung=s["sammlung"], erzaehlung=s["erzaehlung"],
            tranche=s["tranche"], block=s["block"],
            satzindex=s["satzindex"], token_nr=t.i + 1, pronomen=t.text,
            typ="personal" if ist_personal else "demonstrativ",
            satz=" ".join(s["tokens"]),
            satz_markiert=" ".join(markiert),
            block_markiert=block_text,
            referent_gold="", kommentar=""))

from collections import Counter
print(f"Gefundene Pronomen: {len(zeilen)}")
print("  nach Typ:", dict(Counter(z['typ'] for z in zeilen)))
print("  nach Sammlung:", dict(Counter(z['sammlung'] for z in zeilen)))

## Schritt 4 — Sortieren und als CSV schreiben



In [ ]:
import csv

zeilen.sort(key=lambda r: (r["sammlung"], r["erzaehlung"],
                           r["satzindex"], r["token_nr"]))

spalten = ["sammlung", "erzaehlung", "tranche", "block", "satzindex",
           "token_nr", "pronomen", "typ", "satz", "satz_markiert",
           "block_markiert", "referent_gold", "kommentar"]
with open(AUSGABE_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=spalten)
    w.writeheader()
    w.writerows(zeilen)

print(f"Geschrieben: {AUSGABE_CSV}  ({len(zeilen)} Zeilen)")
print("Die Spalte 'referent_gold' ist leer und wird von den "
      "Annotierenden gefüllt.")

#Nutzen des Blockkontexts beziffern
import re
def hat_bezug(text_markiert):
    stelle = re.search(r"\*\*(.+?)\*\*", text_markiert)
    if not stelle:
        return False
    davor = re.findall(r"\w+", text_markiert[:stelle.start()])
    return any(w[:1].isupper() and w.isalpha() and len(w) > 2
               for i, w in enumerate(davor) if i > 0)

nur_satz = sum(1 for z in zeilen if hat_bezug(z["satz_markiert"]))
mit_block = sum(1 for z in zeilen if hat_bezug(z["block_markiert"]))
print(f"\nKandidat fuer den Bezug vor dem Pronomen auffindbar:")
print(f"  im Einzelsatz:      {nur_satz}/{len(zeilen)} "
      f"({100*nur_satz/len(zeilen):.0f} %)")
print(f"  im Blockkontext:    {mit_block}/{len(zeilen)} "
      f"({100*mit_block/len(zeilen):.0f} %)")
print(f"  zusaetzlich geloest: {mit_block - nur_satz}")
print("Die uebrigen Faelle verlangen weiterhin das Nachlesen im Maerchen.")

## Schritt 5 — Kontrolle


In [ ]:
kontrolle = list(csv.DictReader(open(AUSGABE_CSV, encoding="utf-8")))
print(f"Zeilen in der Datei: {len(kontrolle)}")
print("Typen:", dict(Counter(r["typ"] for r in kontrolle)))
print("\nDrei Beispielzeilen:")
for r in kontrolle[:3]:
    print(f"  [{r['sammlung']}/{r['erzaehlung']}] Satz {r['satzindex']}, "
          f"Token {r['token_nr']}: '{r['pronomen']}' ({r['typ']})")
    print(f"      {r['satz_markiert'][:70]}...")